In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error

def run_eda_plots(df):
    print("📊 Running EDA...")

    ts_agg = df.groupby("datetime")["target"].mean().reset_index()
    plt.figure(figsize=(14, 5))
    plt.plot(ts_agg["datetime"], ts_agg["target"])
    plt.title("Average Energy Consumption Over Time")
    plt.xlabel("Datetime")
    plt.ylabel("Target (Energy Consumption)")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(8, 5))
    sns.histplot(df["target"].dropna(), bins=50, kde=True)
    plt.title("Distribution of Energy Consumption (Target)")
    plt.tight_layout()
    plt.show()

    df["hour"] = df["datetime"].dt.hour
    df["dayofweek"] = df["datetime"].dt.dayofweek

    box_hour = df.groupby("hour")["target"].mean().reset_index()
    plt.figure(figsize=(12, 5))
    sns.barplot(x="hour", y="target", data=box_hour)
    plt.title("Average Energy Consumption by Hour of Day")
    plt.tight_layout()
    plt.show()

    box_day = df.groupby("dayofweek")["target"].mean().reset_index()
    plt.figure(figsize=(12, 5))
    sns.barplot(x="dayofweek", y="target", data=box_day)
    plt.title("Average Energy Consumption by Day of Week")
    plt.tight_layout()
    plt.show()

    print("✅ EDA completed.\n")

def load_and_merge_data(train_path, client_path, gas_path, weather_path):
    print("Loading datasets...")
    train = pd.read_csv(train_path, parse_dates=["datetime"])
    client = pd.read_csv(client_path, parse_dates=["date"])
    gas_prices = pd.read_csv(gas_path, parse_dates=["forecast_date"])
    historical_weather = pd.read_csv(weather_path, parse_dates=["datetime"])

    train = train.merge(
    client[['data_block_id', 'installed_capacity', 'is_business', 'product_type', 'county']], 
    on='data_block_id', 
    how='left'
)


    # Merge gas prices
    train = train.merge(
        gas_prices[['data_block_id', 'lowest_price_per_mwh', 'highest_price_per_mwh']], 
        on='data_block_id', 
        how='left'
    )

    # Merge weather (historical)
    train = train.merge(
        historical_weather[['datetime', 'temperature', 'dewpoint', 'rain', 'snowfall', 
                           'surface_pressure', 'cloudcover_total', 'windspeed_10m', 
                           'winddirection_10m', 'shortwave_radiation', 'direct_solar_radiation',
                           'diffuse_radiation']], 
        on='datetime', 
        how='left'
    )

    print("Merge completed.")
    return train

def feature_engineering(df):
    df['hour'] = df['datetime'].dt.hour
    df['dayofweek'] = df['datetime'].dt.dayofweek
    df['month'] = df['datetime'].dt.month
    df['is_weekend'] = (df['dayofweek'] >= 5).astype(int)
    df['is_business'] = df['is_business'].astype(int)

    # One-hot encode product_type
    product_dummies = pd.get_dummies(df['product_type'], prefix='product_type')
    df = pd.concat([df, product_dummies], axis=1)

    # Use windspeed_10m as wind_speed
    df['wind_speed'] = df['windspeed_10m']

    # Sort for lags
    df = df.sort_values(['prediction_unit_id', 'is_consumption', 'datetime'])

    # Create lag features on target
    lags = [1, 2, 3, 24, 48]
    for lag in lags:
        df[f'target_lag_{lag}'] = df.groupby(['prediction_unit_id', 'is_consumption'])['target'].shift(lag)

    # Rolling mean
    df['target_roll_3'] = df.groupby(['prediction_unit_id', 'is_consumption'])['target'].transform(lambda x: x.shift(1).rolling(3).mean())
    df['target_roll_24'] = df.groupby(['prediction_unit_id', 'is_consumption'])['target'].transform(lambda x: x.shift(1).rolling(24).mean())

    lag_cols = [f'target_lag_{lag}' for lag in lags] + ['target_roll_3', 'target_roll_24']
    df[lag_cols] = df[lag_cols].fillna(0)

    feature_cols = [
        'hour', 'dayofweek', 'month', 'is_weekend',
        'installed_capacity', 'is_business',
        'lowest_price_per_mwh', 'highest_price_per_mwh',
        'temperature', 'dewpoint', 'rain', 'snowfall', 'surface_pressure',
        'cloudcover_total', 'wind_speed', 'shortwave_radiation', 'direct_solar_radiation', 'diffuse_radiation'
    ] + list(product_dummies.columns) + lag_cols

    return df, feature_cols

def train_xgboost(df, feature_cols):
    df = df.dropna(subset=['target'])  # drop rows without target

    X = df[feature_cols]
    y = df['target']

    tscv = TimeSeriesSplit(n_splits=5)
    models = []
    val_scores = []

    for fold, (train_idx, val_idx) in enumerate(tscv.split(X)):
        print(f"Training fold {fold+1}...")
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        model = xgb.XGBRegressor(
            objective='reg:squarederror',
            n_estimators=1000,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            tree_method='hist',
            verbosity=1
        )

        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            early_stopping_rounds=50,
            verbose=100
        )

        y_pred = model.predict(X_val)
        rmse = mean_squared_error(y_val, y_pred, squared=False)
        print(f"Fold {fold+1} RMSE: {rmse:.4f}")

        models.append(model)
        val_scores.append(rmse)

    print(f"Average CV RMSE: {np.mean(val_scores):.4f}")
    return models

def main():
    train_path = "train.csv"
    client_path = "client.csv"
    gas_path = "gas_prices.csv"
    weather_path = "historical_weather.csv"

    train = load_and_merge_data(train_path, client_path, gas_path, weather_path)

    run_eda_plots(train)

    train, feature_cols = feature_engineering(train)

    models = train_xgboost(train, feature_cols)

    print("Training complete.")

if __name__ == "__main__":
    main()
